# CET-Epi: Data Exploration

Explore Hungary Chickenpox dataset and understand multi-scale structure.

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch_geometric_temporal.dataset import ChickenpoxDatasetLoader
from src.data.chickenpox_loader import MultiScaleChickenpoxLoader
from src.utils.config import load_config

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

## 1. Load Raw Dataset

In [ ]:
# Load using PyTorch Geometric Temporal
loader = ChickenpoxDatasetLoader()
raw_dataset = loader.get_dataset(lags=4)

print(f"Dataset type: {type(raw_dataset)}")

# Convert to list to inspect
snapshots = list(raw_dataset)
print(f"Number of temporal snapshots: {len(snapshots)}")
print(f"First snapshot: {type(snapshots[0])}")

## 2. Inspect Graph Structure

In [ ]:
# Get first snapshot
sample = snapshots[0]

print("Snapshot structure:")
print(f"  Node features (x): {sample.x.shape}")
print(f"  Edge index: {sample.edge_index.shape}")
print(f"  Edge attr: {sample.edge_attr.shape if sample.edge_attr is not None else None}")
print(f"  Target (y): {sample.y.shape}")

n_nodes = sample.x.shape[0]
n_edges = sample.edge_index.shape[1]

print(f"\nGraph statistics:")
print(f"  Nodes (counties): {n_nodes}")
print(f"  Edges: {n_edges}")
print(f"  Average degree: {n_edges / n_nodes:.2f}")

## 3. Visualize Graph Connectivity

In [ ]:
import networkx as nx

# Create NetworkX graph
G = nx.Graph()
G.add_nodes_from(range(n_nodes))

edges = sample.edge_index.t().numpy()
for i in range(edges.shape[0]):
    u, v = edges[i]
    if u != v:
        G.add_edge(int(u), int(v))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Circular layout
pos_circular = nx.circular_layout(G)
nx.draw(G, pos_circular, ax=axes[0], node_size=300, node_color='skyblue', 
        with_labels=True, font_size=8, edge_color='gray', alpha=0.7)
axes[0].set_title('Hungary Counties: Circular Layout')

# Spring layout
pos_spring = nx.spring_layout(G, seed=42)
nx.draw(G, pos_spring, ax=axes[1], node_size=300, node_color='lightcoral', 
        with_labels=True, font_size=8, edge_color='gray', alpha=0.7)
axes[1].set_title('Hungary Counties: Spring Layout')

plt.tight_layout()
plt.savefig('../logs/figures/data_graph_structure.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Graph density: {nx.density(G):.3f}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Average clustering: {nx.average_clustering(G):.3f}")

## 4. Temporal Dynamics Analysis

In [ ]:
# Extract time series for all nodes
all_cases = []
for snap in snapshots:
    all_cases.append(snap.y.numpy())

cases_matrix = np.array(all_cases)  # [T, N]

print(f"Time series shape: {cases_matrix.shape}")
print(f"Time range: {cases_matrix.shape[0]} weeks")

# Plot aggregate
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total cases over time
total_cases = cases_matrix.sum(axis=1)
axes[0, 0].plot(total_cases, linewidth=2, color='darkblue')
axes[0, 0].set_title('Total Chickenpox Cases (All Counties)')
axes[0, 0].set_xlabel('Week')
axes[0, 0].set_ylabel('Cases')
axes[0, 0].grid(True, alpha=0.3)

# Heatmap of all counties
im = axes[0, 1].imshow(cases_matrix.T, aspect='auto', cmap='YlOrRd')
axes[0, 1].set_title('Cases Heatmap (Counties x Time)')
axes[0, 1].set_xlabel('Week')
axes[0, 1].set_ylabel('County')
plt.colorbar(im, ax=axes[0, 1])

# Sample counties time series
sample_counties = [0, 5, 10, 15, 19]
for i in sample_counties:
    axes[1, 0].plot(cases_matrix[:, i], label=f'County {i}', alpha=0.7)
axes[1, 0].set_title('Sample Counties Time Series')
axes[1, 0].set_xlabel('Week')
axes[1, 0].set_ylabel('Cases')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Distribution of cases
axes[1, 1].hist(cases_matrix.flatten(), bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Distribution of Case Counts')
axes[1, 1].set_xlabel('Cases')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../logs/figures/data_temporal_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nStatistics:")
print(f"  Mean cases per week: {total_cases.mean():.1f}")
print(f"  Std cases per week: {total_cases.std():.1f}")
print(f"  Peak week: {total_cases.argmax()} ({total_cases.max():.0f} cases)")
print(f"  Min week: {total_cases.argmin()} ({total_cases.min():.0f} cases)")

## 5. Multi-Scale Structure Design

In [ ]:
# Design macro-scale partition
# Strategy 1: Geographic clustering (k-means on coordinates)
# Strategy 2: Graph spectral clustering
# Strategy 3: Data-driven (CEO-learned)

from sklearn.cluster import KMeans, SpectralClustering

# Use spring layout coordinates as proxy for geography
coords = np.array([pos_spring[i] for i in range(n_nodes)])

# K-means clustering
n_macro = 5
kmeans = KMeans(n_clusters=n_macro, random_state=42, n_init=10)
geo_partition = kmeans.fit_predict(coords)

# Spectral clustering
adj_matrix = nx.to_numpy_array(G)
spectral = SpectralClustering(n_clusters=n_macro, affinity='precomputed', random_state=42)
# Convert to similarity
similarity = adj_matrix + np.eye(n_nodes)  # Add self-loops
spectral_partition = spectral.fit_predict(similarity)

# Visualize partitions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_geo = plt.cm.tab10(geo_partition)
nx.draw(G, pos_spring, ax=axes[0], node_color=colors_geo, 
        node_size=400, with_labels=True, font_size=8, edge_color='gray')
axes[0].set_title(f'Geographic Clustering (K={n_macro})')

colors_spec = plt.cm.tab10(spectral_partition)
nx.draw(G, pos_spring, ax=axes[1], node_color=colors_spec, 
        node_size=400, with_labels=True, font_size=8, edge_color='gray')
axes[1].set_title(f'Spectral Clustering (K={n_macro})')

plt.tight_layout()
plt.savefig('../logs/figures/data_clustering_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Cluster sizes (Geographic):", np.bincount(geo_partition))
print("Cluster sizes (Spectral):", np.bincount(spectral_partition))

## 6. Feature Analysis

In [ ]:
# Analyze lag features
sample_features = sample.x.numpy()  # [N, lags, F]

print(f"Feature tensor shape: {sample_features.shape}")
print(f"  N = {sample_features.shape[0]} nodes")
print(f"  Lags = {sample_features.shape[1]} historical weeks")
print(f"  F = {sample_features.shape[2]} features per lag")

# Feature correlation
feat_flat = sample_features.reshape(-1, sample_features.shape[-1])
corr_matrix = np.corrcoef(feat_flat.T)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.savefig('../logs/figures/data_feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# Autocorrelation in time series
from statsmodels.tsa.stattools import acf

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i in range(min(6, n_nodes)):
    autocorr = acf(cases_matrix[:, i], nlags=20, fft=True)
    axes[i].plot(autocorr, marker='o')
    axes[i].set_title(f'County {i} Autocorrelation')
    axes[i].set_xlabel('Lag')
    axes[i].set_ylabel('Correlation')
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(y=0, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../logs/figures/data_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Prepare Multi-Scale Loader

In [ ]:
# Initialize our custom loader
custom_loader = MultiScaleChickenpoxLoader(lags=4)
train_data, test_data = custom_loader.get_split(train_ratio=0.8)

print(f"Train snapshots: {len(list(train_data))}")
print(f"Test snapshots: {len(list(test_data))}")

# Get static graph
edge_index, edge_attr = custom_loader.get_static_graph()
print(f"\nStatic graph:")
print(f"  Edge index shape: {edge_index.shape}")
print(f"  Edge attr shape: {edge_attr.shape if edge_attr is not None else None}")

# Test a batch
train_list = list(train_data)
batch = train_list[0]
print(f"\nBatch structure:")
print(f"  x: {batch.x.shape}")
print(f"  y: {batch.y.shape}")

print("\n✅ Data exploration complete! Ready for model training.")